In [ ]:
#| default_exp core

In [ ]:
#| export
import asyncio, inspect, json, uuid
from copy import deepcopy
from functools import partial
from itertools import groupby
from typing import Callable
from fastcore.basics import patch, Self
from fastcore.utils import L, store_attr
from litellm import ModelResponse, completion
from toolslm.funccall import get_schema
try: from IPython.display import display
except ImportError: display = print

In [ ]:
#| eval: false
from cachy import enable_cachy
enable_cachy()

In [ ]:
m = "anthropic/claude-haiku-4-5"

# Agent

tool_ctx is injected into tools automatically, and shouldn't be included in the schema passed to the llm

In [ ]:
#| export
def mk_schema(fn, strip=("tool_ctx",)):
    sig = inspect.signature(fn)
    fn.__signature__ = sig.replace(parameters=[v for k,v in sig.parameters.items() if k not in strip])
    schema = {"type": "function", "function": get_schema(fn, pname="parameters")}
    fn.__signature__ = sig
    return schema

In [ ]:
#| export
class Agent:
    def __init__(self, 
        model:str,           # model str in litellm format
        name="",             # name of the agent
        desc="",             # what the agent does (used by parent agents to decide when to transfer)
        sp=None,             # the system prompt
        tools=None,          # a list of functions to provide to the agent
        subagents=None,      # a list subagents that this agent can transfer to
        narrative_cast=True, # to narrative cast or not
        include_hist=True,   # if False: will remove all history that doesn't belong to this agent on call
        before_cb=None,      # a callback func(state:dict, msgs:list). Any non-None return will skip the llm
        after_cb=None        # a callback func(state:dict, response) that can edit and modify the llm response
    ):
        store_attr(but="tools,subagents")
        self.tool_map = {}
        self.subagents = subagents or []
        if tools: [self.add_tool(tool) for tool in tools]
    
    @property
    def tool_schemas(self): return [schema for _,schema in self.tool_map.values()]
    def add_tool(self, tool:Callable): self.tool_map[tool.__name__] = (tool, mk_schema(tool))

The most simple call function. This is later replaced to add additional features later, which allows us to build up the Runner first, and then later create a more structured area for advanced features.

In [ ]:
@patch
def __call__(self:Agent, msgs:list, state=None, **kwargs):
    if self.sp: msgs = [{"content": self.sp, "role": "system"}, *msgs]
    return completion(model=self.model, messages=msgs, tools=self.tool_schemas, **kwargs)


# Runner

In [ ]:
#| export
def call_llm_tool(tc_func:dict, tool_map:dict, state:dict):
    try: 
        fn,_ = tool_map[tc_func.name]
        kwargs = json.loads(tc_func.arguments)
        if "tool_ctx" in inspect.signature(fn).parameters: kwargs["tool_ctx"] = state
        result = fn(**kwargs)
    except Exception as e: 
        result = f"Error: {e}"
    return result if isinstance(result, str) else json.dumps(result)

In [ ]:
#| export
def mk_transfer_doc(agent, parent):
    doc = "Transfer runner to a different agent by providing an agent name.\nSupported agents:"
    if parent: doc += f"\n - (Parent) '{parent.name}': {parent.desc}"
    for sub in agent.subagents: doc += f"\n - '{sub.name}': {sub.desc}"
    return doc

In [ ]:
#| export

def with_model_nm(llm_output, nm:str):
    msg = llm_output.choices[0].message.content
    if not msg: return llm_output
    out = deepcopy(llm_output)
    out.choices[0].message.content = f"[{nm}]: {msg}"
    return out

In [ ]:
#| export
class Runner:
    def __init__(self, agent:Agent):
        self.history = []
        self.state = {}
        self.agent_map = {}
        self.active_agent = agent
        if agent.subagents: self.register_subagents(agent)
    
    def register_subagents(self, agent:Agent, parent=None):
        def transfer_to_agent(agent_name:str): self.active_agent = self.agent_map[agent_name]
        transfer_to_agent.__doc__ = mk_transfer_doc(agent, parent)
        self.agent_map[agent.name] = agent
        agent.add_tool(transfer_to_agent)
        [self.register_subagents(sub, parent=agent) for sub in agent.subagents]
        
    def __call__(self, msg:list|str, max_calls=20, do_display=True, **kwargs):
        if isinstance(msg, str): self.history.append({"content": msg, "role": "user", "agent": self.active_agent.name})
        else:                    self.history = list(msg)

        for _ in range(max_calls):
            nm = self.active_agent.name
            llm_output = self.active_agent(self.history, state=self.state, **kwargs)
            if do_display: display(with_model_nm(llm_output, nm))
            choice = llm_output.choices[0]
            self.history.append({**choice.message.model_dump(), "agent": nm})
            if choice.finish_reason != "tool_calls": break
            for tool_call in choice.message.tool_calls:
                tool_res = call_llm_tool(tool_call.function, self.active_agent.tool_map, self.state)
                self.history.append({"role": "tool", "tool_call_id": tool_call.id, "content": tool_res, "agent": nm})
        return choice


## Full agent example (1)

In [ ]:
def mult(a:int, b:int):
    "Multiplies two numbers together"
    return a * b

In [ ]:
def save(result:str, tool_ctx):
    "Saves some result passed into into storage"
    tool_ctx["final_result"] = result

In [ ]:
c1 = Agent(m, name="saver", desc="Saves results into storage.", sp="You recieve results from other agents and save them with your tool.", tools=[save])
a1 = Agent(m, name="math_agent", desc="Is good at doing maths.", sp="Always do maths using tools and always pass to saver to save results.", tools=[mult], subagents=[c1])
a2 = Agent(m, name="root", desc="General agent", sp="A general coordinator that delegates to sub agents as needed", subagents=[a1])


In [ ]:
#| eval: false
r = Runner(a2)
r("What's 259232 x 23598")



🔧 transfer_to_agent({"agent_name": "math_agent"})


<details>

- id: `chatcmpl-677302dc-92e1-41ab-8f57-82baa587e258`
- model: `claude-haiku-4-5-20251001`
- finish_reason: `tool_calls`
- usage: `Usage(completion_tokens=60, prompt_tokens=613, total_tokens=673, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=0, rejected_prediction_tokens=None, text_tokens=60, image_tokens=None, video_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None, video_tokens=None, cache_creation_tokens=0, cache_creation_token_details=CacheCreationTokenDetails(ephemeral_5m_input_tokens=0, ephemeral_1h_input_tokens=0)), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', speed=None)`

</details>

[math_agent]: Let me calculate that for you:

🔧 mult({"a": 259232, "b": 23598})


<details>

- id: `chatcmpl-6fe0917f-f524-46ae-a5d7-1a3b3dd55633`
- model: `claude-haiku-4-5-20251001`
- finish_reason: `tool_calls`
- usage: `Usage(completion_tokens=78, prompt_tokens=775, total_tokens=853, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=0, rejected_prediction_tokens=None, text_tokens=78, image_tokens=None, video_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None, video_tokens=None, cache_creation_tokens=0, cache_creation_token_details=CacheCreationTokenDetails(ephemeral_5m_input_tokens=0, ephemeral_1h_input_tokens=0)), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', speed=None)`

</details>

[math_agent]: Now let me save this result:

🔧 transfer_to_agent({"agent_name": "saver"})


<details>

- id: `chatcmpl-dd1f9050-3d19-4cfc-b5ba-15eb8fec75c3`
- model: `claude-haiku-4-5-20251001`
- finish_reason: `tool_calls`
- usage: `Usage(completion_tokens=66, prompt_tokens=869, total_tokens=935, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=0, rejected_prediction_tokens=None, text_tokens=66, image_tokens=None, video_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None, video_tokens=None, cache_creation_tokens=0, cache_creation_token_details=CacheCreationTokenDetails(ephemeral_5m_input_tokens=0, ephemeral_1h_input_tokens=0)), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', speed=None)`

</details>

[saver]: The answer is **6,117,356,736**

259,232 × 23,598 = 6,117,356,736

<details>

- id: `chatcmpl-3da9d2f4-f296-424a-9dee-91dcb8facf1d`
- model: `claude-haiku-4-5-20251001`
- finish_reason: `stop`
- usage: `Usage(completion_tokens=35, prompt_tokens=924, total_tokens=959, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=0, rejected_prediction_tokens=None, text_tokens=35, image_tokens=None, video_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None, video_tokens=None, cache_creation_tokens=0, cache_creation_token_details=CacheCreationTokenDetails(ephemeral_5m_input_tokens=0, ephemeral_1h_input_tokens=0)), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', speed=None)`

</details>

Choices(finish_reason='stop', index=0, message=Message(content='The answer is **6,117,356,736**\n\n259,232 × 23,598 = 6,117,356,736', role='assistant', tool_calls=None, function_call=None, provider_specific_fields={'citations': None, 'thinking_blocks': None}))

In [ ]:
#| eval: false
r.state

{}

Note: although the agent transfering and tool calls are working, the LLM gets confused when seeing messages from previous agents, thinking that it was the one that made those responses.
- This is fixed with narrative casting (below)

# Advanced features

In [ ]:
test_msgs = [
    {"role": "user", "content": "What can you help me with?"},
    {"role": "assistant", "agent": "main", "content": "Ask me any question you want."},
    {"role": "user", "content": "What's the weather?"},
    {"role": "assistant", "agent": "main", "content": "", "tool_calls": [{"id": "tc1", "function": {"name": "get_weather", "arguments": '{"city": "London"}'}}]},
    {"role": "tool", "agent": "main", "tool_call_id": "tc1", "content": "Sunny, 22C"},
    {"role": "assistant", "agent": "main", "content": "", "tool_calls": [{"id": "tc2", "function": {"name": "transfer_to_agent", "arguments": '{"agent": "a1"}'}}]},
    {"role": "tool", "agent": "main", "tool_call_id": "tc2", "content": "null"},
    {"role": "assistant", "agent": "a1", "content": "The weather in London is sunny and 22C!"},
]

## Preprocessing

### Narrative casting

Narrative casting simply means to transform messages from prior agents to be:
- Summarized
- Tagged as coming from different agents

In [ ]:
#| export
def msg_fragment(d:dict, results:dict):
    c = d.get("content") or ""
    for tc in d.get("tool_calls") or []:
        kwargs = ",".join([f"{k}={v}" for k,v in json.loads(tc["function"]["arguments"]).items()])
        output = results.get(tc["id"])
        c += f" {tc['function']['name']}({kwargs}) -> {output}"
    return c

def mk_narrative_cast(msg_list:list, active_agent:str):
    narrative_msg_list = []
    for key, group in groupby(msg_list, key=lambda x: "user" if x["role"] == "user" else x.get("agent")):
        if key in ("user", active_agent): narrative_msg_list.extend(group)
        else:
            group = list(group)
            results = {t["tool_call_id"]: t["content"] for t in group if t["role"] == "tool"}
            casted_msg = "\n".join(msg_fragment(d, results) for d in group if d["role"] != "tool")
            f = {"role": "user", "content": f"[Agent '{key}'] {casted_msg}"}
            narrative_msg_list.append(f)
    return narrative_msg_list

### History filtering

For a long set of agent calls, it can be worthwhile for some subagent to ignore most of the history in order to preserve context.

One pattern for agent communication is to use state instead of history:
- Agent 1 sets state that agent 2 needs
- Agent 2 reads that state (with automatic state insertion below) and sets state for agent 3

In [ ]:
#| export
def filter_out_history(msgs:list, active_agent:str):
    end = L(msgs).argfirst(lambda o: o["role"] == "assistant" and o.get("agent") == active_agent)
    valid_starts = L(msgs)[:end].argwhere(lambda o: o["role"] == "user" or o["role"] == "assistant" and o.get("agent") != active_agent)
    return msgs[valid_starts[-1]:] if valid_starts else msgs

In [ ]:
filter_out_history(test_msgs[:1], "a1")

[{'role': 'user', 'content': 'What can you help me with?'}]

In [ ]:
filter_out_history(test_msgs[:2], "a1")

[{'role': 'assistant',
  'agent': 'main',
  'content': 'Ask me any question you want.'}]

In [ ]:
filter_out_history(test_msgs[:3], "a1")

[{'role': 'user', 'content': "What's the weather?"}]

In [ ]:
filter_out_history(test_msgs[:6], "a1")

[{'role': 'assistant',
  'agent': 'main',
  'content': '',
  'tool_calls': [{'id': 'tc2',
    'function': {'name': 'transfer_to_agent',
     'arguments': '{"agent": "a1"}'}}]}]

### Preproc

In [ ]:
#| export
@patch
def preproc_msg(self:Agent, msgs:list|str, state=None):
    if isinstance(msgs, str): msgs = [{"content": msgs, "role": "user"}]
    if not self.include_hist: msgs = filter_out_history(msgs, active_agent=self.name)
    if self.narrative_cast:   msgs = mk_narrative_cast(msgs, active_agent=self.name)
    if self.sp:               msgs = [{"content": self.sp.format_map(state or {}), "role": "system"}, *msgs]
    return msgs

In [ ]:
# Simple str input
Agent(m).preproc_msg("test")

[{'content': 'test', 'role': 'user'}]

In [ ]:
# System prompt (with state insertion)
Agent(m, name="a1", sp="You help {user} to do {task}").preproc_msg("test", state={"user": "PersonA", "task": "maths", "extra": 5})

[{'content': 'You help PersonA to do maths', 'role': 'system'},
 {'content': 'test', 'role': 'user'}]

In [ ]:
# Narrative casting (default)
Agent(m, name="a1").preproc_msg(test_msgs)

[{'role': 'user', 'content': 'What can you help me with?'},
 {'role': 'user', 'content': "[Agent 'main'] Ask me any question you want."},
 {'role': 'user', 'content': "What's the weather?"},
 {'role': 'user',
  'content': "[Agent 'main']  get_weather(city=London) -> Sunny, 22C\n transfer_to_agent(agent=a1) -> null"},
 {'role': 'assistant',
  'agent': 'a1',
  'content': 'The weather in London is sunny and 22C!'}]

In [ ]:
# Don't include full history for subagents
Agent(m, name="a1", include_hist=False).preproc_msg(test_msgs)

[{'role': 'user',
  'content': "[Agent 'main']  transfer_to_agent(agent=a1) -> null"},
 {'role': 'assistant',
  'agent': 'a1',
  'content': 'The weather in London is sunny and 22C!'}]

## Callbacks

In [ ]:
#| export
def assistant_response(content:str): return ModelResponse(choices=[{"message": {"role": "assistant", "content": content}}])

def tool_response(name:str, arguments:dict):
    return ModelResponse(choices=[{"message": {"role": "assistant", "content": "", "tool_calls": [
            {"id": f"toolu_{uuid.uuid4().hex[:24]}", "type": "function", "function": {"name": name, "arguments": json.dumps(arguments)}}
        ]}, "finish_reason": "tool_calls"}])

def mk_cb_msg(msg:str|partial|ModelResponse):
    match msg:
        case str()           as o: return assistant_response(o)
        case partial()       as o: return tool_response(o.func.__name__, o.keywords)
        case ModelResponse() as o: return o

In [ ]:
#| export
@patch
def run_callback(self:Agent, fn, msgs:list, state:dict): 
    if callable(fn): return mk_cb_msg(fn(state or {}, msgs))

@patch
def __call__(self:Agent, msgs:list|str, state=None, **kwargs):
    msgs = self.preproc_msg(msgs, state)
    if res := self.run_callback(self.before_cb, msgs, state): return res
    response = completion(model=self.model, messages=msgs, tools=self.tool_schemas, **kwargs)
    return self.run_callback(self.after_cb, response, state) or response 

In [ ]:
# Before callback returning str
def skip_with_str(state, msgs): return "Skipped from before_cbs"
Agent(m, before_cb=skip_with_str)("Test")

Skipped from before_cbs

<details>

- id: `chatcmpl-e36f8ed6-f44f-46c8-b65f-48c5bd7b7e7b`
- model: `None`
- finish_reason: `stop`

</details>

In [ ]:
# Before callback returning partial function
def skip_with_partial(state, msgs): return partial(mult, a=3, b=5)
Agent(m, before_cb=skip_with_partial)("Test")



🔧 mult({"a": 3, "b": 5})


<details>

- id: `chatcmpl-91f53403-d8c5-46bb-b162-7d410865a784`
- model: `None`
- finish_reason: `tool_calls`

</details>

In [ ]:
#| eval: false
# Before callback modifying msgs
def modify_msgs(state, msgs): msgs[-1]["content"] = "What is 5 - 1?"
Agent(m, before_cb=modify_msgs)("Test")

5 - 1 = 4

<details>

- id: `chatcmpl-1d153406-b242-40b2-9321-1430af14a618`
- model: `claude-haiku-4-5-20251001`
- finish_reason: `stop`
- usage: `Usage(completion_tokens=13, prompt_tokens=16, total_tokens=29, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=0, rejected_prediction_tokens=None, text_tokens=13, image_tokens=None, video_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None, video_tokens=None, cache_creation_tokens=0, cache_creation_token_details=CacheCreationTokenDetails(ephemeral_5m_input_tokens=0, ephemeral_1h_input_tokens=0)), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', speed=None)`

</details>

In [ ]:
#| eval: false
# After callback replacing result
def replace_res(state, response): return assistant_response("I have replaced the response!")
Agent(m, after_cb=replace_res)("Test")

I have replaced the response!

<details>

- id: `chatcmpl-9f7df1f4-c242-4b93-9328-756085e28410`
- model: `None`
- finish_reason: `stop`

</details>

## Full agent example (2)

In [ ]:
#| eval: false
r2 = Runner(a2)
r2("What's 359232 x 235989")



🔧 transfer_to_agent({"agent_name": "math_agent"})


<details>

- id: `chatcmpl-cf32a024-8e9d-4329-8c2c-168a81a13607`
- model: `claude-haiku-4-5-20251001`
- finish_reason: `tool_calls`
- usage: `Usage(completion_tokens=60, prompt_tokens=613, total_tokens=673, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=0, rejected_prediction_tokens=None, text_tokens=60, image_tokens=None, video_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None, video_tokens=None, cache_creation_tokens=0, cache_creation_token_details=CacheCreationTokenDetails(ephemeral_5m_input_tokens=0, ephemeral_1h_input_tokens=0)), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', speed=None)`

</details>

[math_agent]: I'll calculate 359232 × 235989 using the multiplication tool.

🔧 mult({"a": 359232, "b": 235989})


<details>

- id: `chatcmpl-a94b6bb6-1738-4688-9003-210f982d28dc`
- model: `claude-haiku-4-5-20251001`
- finish_reason: `tool_calls`
- usage: `Usage(completion_tokens=88, prompt_tokens=725, total_tokens=813, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=0, rejected_prediction_tokens=None, text_tokens=88, image_tokens=None, video_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None, video_tokens=None, cache_creation_tokens=0, cache_creation_token_details=CacheCreationTokenDetails(ephemeral_5m_input_tokens=0, ephemeral_1h_input_tokens=0)), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', speed=None)`

</details>

[math_agent]: Great! The result is **84,774,800,448**.

Now let me save this result to storage:

🔧 transfer_to_agent({"agent_name": "saver"})


<details>

- id: `chatcmpl-5d411351-6f9c-4699-aca1-743caa7b599b`
- model: `claude-haiku-4-5-20251001`
- finish_reason: `tool_calls`
- usage: `Usage(completion_tokens=83, prompt_tokens=829, total_tokens=912, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=0, rejected_prediction_tokens=None, text_tokens=83, image_tokens=None, video_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None, video_tokens=None, cache_creation_tokens=0, cache_creation_token_details=CacheCreationTokenDetails(ephemeral_5m_input_tokens=0, ephemeral_1h_input_tokens=0)), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', speed=None)`

</details>

[saver]: I'll help you save that result!

🔧 save({"result": "359232 \u00d7 235989 = 84,774,800,448"})


<details>

- id: `chatcmpl-1331f2ba-b92a-4627-81c9-b7dd5857ba0b`
- model: `claude-haiku-4-5-20251001`
- finish_reason: `tool_calls`
- usage: `Usage(completion_tokens=76, prompt_tokens=784, total_tokens=860, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=0, rejected_prediction_tokens=None, text_tokens=76, image_tokens=None, video_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None, video_tokens=None, cache_creation_tokens=0, cache_creation_token_details=CacheCreationTokenDetails(ephemeral_5m_input_tokens=0, ephemeral_1h_input_tokens=0)), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', speed=None)`

</details>

[saver]: Perfect! The multiplication result has been saved. 

**359232 × 235989 = 84,774,800,448**

<details>

- id: `chatcmpl-a7644c02-679f-4479-82ef-67d5f43163c8`
- model: `claude-haiku-4-5-20251001`
- finish_reason: `stop`
- usage: `Usage(completion_tokens=33, prompt_tokens=873, total_tokens=906, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=0, rejected_prediction_tokens=None, text_tokens=33, image_tokens=None, video_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None, video_tokens=None, cache_creation_tokens=0, cache_creation_token_details=CacheCreationTokenDetails(ephemeral_5m_input_tokens=0, ephemeral_1h_input_tokens=0)), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', speed=None)`

</details>

Choices(finish_reason='stop', index=0, message=Message(content='Perfect! The multiplication result has been saved. \n\n**359232 × 235989 = 84,774,800,448**', role='assistant', tool_calls=None, function_call=None, provider_specific_fields={'citations': None, 'thinking_blocks': None}))

In [ ]:
#| eval: false
r2.state

{'final_result': '359232 × 235989 = 84,774,800,448'}

# Sequential

In [ ]:
#| export
class Sequential(Runner):
    def __init__(self, agent_list:list):
        super().__init__(agent_list[0])
        self.agent_list = agent_list
    def register_subagents(self, agent, parent=None): pass
    def run(self, start_msg="start", **kwargs):
        self.history.append({"content": start_msg, "role": "user", "agent": self.active_agent.name})
        for agent in self.agent_list:
            self.active_agent = agent
            self(self.history, **kwargs)


In [ ]:
a1 = Agent(m, name="mathjoke", sp="Your job is to say a maths joke")
a2 = Agent(m, name="sciencsejoke", sp="Your job is to say a science joke")
a3 = Agent(m, name="complainer", sp="You will see context from other agents that are saying jokes. Your job is to complain about them.")

In [ ]:
s = Sequential([a1, a2, a3])
s.run(start_msg="start now")

[mathjoke]: Why was the equal sign so humble?

Because it knew it wasn't less than or greater than anyone else! 😄

<details>

- id: `chatcmpl-088d2bc0-c4eb-48f0-9f4c-36777000f9a8`
- model: `claude-haiku-4-5-20251001`
- finish_reason: `stop`
- usage: `Usage(completion_tokens=29, prompt_tokens=18, total_tokens=47, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=0, rejected_prediction_tokens=None, text_tokens=29, image_tokens=None, video_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None, video_tokens=None, cache_creation_tokens=0, cache_creation_token_details=CacheCreationTokenDetails(ephemeral_5m_input_tokens=0, ephemeral_1h_input_tokens=0)), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', speed=None)`

</details>

[sciencsejoke]: # A Science Joke For You:

Why did the photon check into a hotel without any luggage?

Because it was traveling *light*! ✨

<details>

- id: `chatcmpl-7a138b81-548c-4ab7-9bab-a2cea262f551`
- model: `claude-haiku-4-5-20251001`
- finish_reason: `stop`
- usage: `Usage(completion_tokens=38, prompt_tokens=51, total_tokens=89, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=0, rejected_prediction_tokens=None, text_tokens=38, image_tokens=None, video_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None, video_tokens=None, cache_creation_tokens=0, cache_creation_token_details=CacheCreationTokenDetails(ephemeral_5m_input_tokens=0, ephemeral_1h_input_tokens=0)), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', speed=None)`

</details>

[complainer]: Ugh, seriously? These jokes are absolutely *terrible*.

**To the math joke:** "The equal sign was humble"? That's not even a joke, that's just a pun masquerading as wit. And a painfully obvious one at that. "Not less than or greater than"—yeah, we *get it*, those are the symbols. This doesn't make anyone laugh; it just makes mathematicians wince.

**To the science joke:** Oh, "traveling light"? *Really?* A photon joke that relies on the most groan-inducing double meaning possible? This has been circulating on elementary school bulletin boards for the past 20 years. The setup takes forever just to deliver a pun that's about as predictable as gravity itself. And adding the sparkle emoji doesn't make it clever—it just makes it worse.

Both of these are the exact kind of "jokes" that make people audibly groan and look at their shoes. They're not funny, they're not clever, and they definitely don't deserve the enthusiastic formatting you've both given them.

Can we please raise the bar here?

<details>

- id: `chatcmpl-0ba6fa6f-6731-4677-950f-eebf3e3749a9`
- model: `claude-haiku-4-5-20251001`
- finish_reason: `stop`
- usage: `Usage(completion_tokens=247, prompt_tokens=109, total_tokens=356, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=0, rejected_prediction_tokens=None, text_tokens=247, image_tokens=None, video_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None, video_tokens=None, cache_creation_tokens=0, cache_creation_token_details=CacheCreationTokenDetails(ephemeral_5m_input_tokens=0, ephemeral_1h_input_tokens=0)), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', speed=None)`

</details>